In [2]:
import json

import polars as pl
import requests


def load_club_data(file):
    """
    Loads club data from a CSV file.

    Arguments:
    - file (str): The file path of the CSV file.
    - rows (int): The number of rows to load from the file.

    Returns:
    - data (pd.DataFrame): The loaded club data as a pandas DataFrame.
    """
    data = pl.read_csv(file)
    return data


data = load_club_data("postcodes_master_2025.csv")

In [3]:
def get_distance(point1: dict, point2: dict) -> tuple:
    """
    Gets the driving distance and duration between two points using
    http://project-osrm.org/docs/v5.10.0/api/#nearest-service

    Arguments:
    - point1 (dict): A dictionary representing the latitude and longitude of the first point.
    - point2 (dict): A dictionary representing the latitude and longitude of the second point.

    Returns:
    - tuple: A tuple containing the distance (in meters) and duration (in seconds) of the route.
    """
    url = (
            "http://router.project-osrm.org/route/v1/driving/"
            f"{point1['longitude']},{point1['latitude']};"
            f"{point2['longitude']},{point2['latitude']}"
            "?overview=false&alternatives=false"
        )
    r = requests.get(url)

    # get the distance from the returned values
    route = json.loads(r.content)["routes"][0]
    return (route["distance"], route["duration"])

In [ ]:
def create_dist_array(step: str):
    """
    Creates an array of distances between all combinations of points *within the same league*.

    Arguments:
    - step (str): The league step to filter the dataset by (e.g. 'Step 3').

    Returns:
    - dist_array (list): A list of tuples containing the origin index, destination index,
    duration (in seconds), and distance (in meters) between each pair of points.
    """
    dist_array = []

    # Filter the dataset by the given step
    step_data = data.filter(pl.col("step") == step)

    for i, r in step_data.iter_rows(named=True):
        point1 = {"latitude": r["latitude"], "longitude": r["longitude"]}
        league = r["league"]

        # Only compare with other teams in the same league
        same_league_df = step_data[(step_data.index != i) & (step_data["league"] == league)]

        for j, o in same_league_df.iter_rows(named=True):
            point2 = {"latitude": o["latitude"], "longitude": o["longitude"]}
            dist, duration = get_distance(point1, point2)
            dist_array.append((i, j, duration, dist))

    return dist_array

dist_array = create_dist_array("Step 1")

ValueError: too many values to unpack (expected 2)

In [16]:
step_data

team,ground,postcode,county_fa,latitude,longitude,league,league_key,step
str,str,str,str,f64,f64,str,i64,str
"""Aldershot Town""","""EBB Stadium""","""GU11 1TW""","""Hampshire""",51.248865,-0.755051,"""National League""",5,"""Step 1"""
"""Altrincham""","""J Davidson Stadium""","""WA15 8AP""","""Cheshire""",53.382975,-2.335214,"""National League""",5,"""Step 1"""
"""Boreham Wood""","""LV Bet Stadium""","""WD6 5AL""","""Hertfordshire""",51.662119,-0.273709,"""National League""",5,"""Step 1"""
"""Boston United""","""Jakemans Community Stadium""","""PE21 7NE""","""Lincolnshire""",52.955515,-0.030786,"""National League""",5,"""Step 1"""
"""Brackley Town""","""St. James Park""","""NN13 7EJ""","""Northamptonshire""",52.02523,-1.146971,"""National League""",5,"""Step 1"""
…,…,…,…,…,…,…,…,…
"""Tamworth""","""CR MOT Centre Community Stadiu…","""B77 1AA""","""Birmingham""",52.629023,-1.687035,"""National League""",5,"""Step 1"""
"""Truro City""","""Bolitho Park""","""PL5 3JG""","""Cornwall""",50.409774,-4.145733,"""National League""",5,"""Step 1"""
"""Wealdstone""","""Grosvenor Vale""","""HA4 6JQ""","""Middlesex""",51.569018,-0.419526,"""National League""",5,"""Step 1"""


In [ ]:
for i, r in step_data.iter_rows(named=True):
    point1 = {"latitude": r["latitude"], "longitude": r["longitude"]}
    league = r["league"]
        print(league)

ValueError: too many values to unpack (expected 2)

In [7]:
def create_distances_df():
    """
    Creates a DataFrame of distances between all combinations of points.

    Returns:
    - distances_df (pd.DataFrame): The DataFrame containing the distances between each pair of points,
    including origin and destination names, distance in miles, duration in HH:MM:SS format, and a fixture key.
    """

    distances_df = pd.DataFrame(dist_array, columns=["origin", "destination", "duration(s)", "distance(m)"])
    distances_df = distances_df.merge(data[["team","league"]], left_on="origin", right_index=True).rename(
        columns={"team": "origin_name", "league": "origin_league"},
    )
    distances_df = distances_df.merge(data[["team", "league"]], left_on="destination", right_index=True).rename(
        columns={"team": "destination_name", "league": "destination_league"},
    )

    distances_df = distances_df[
    distances_df["origin_league"] == distances_df["destination_league"]].reset_index(drop=True)
    
    distances_df["distance(miles)"] = distances_df["distance(m)"] * 0.000621371
    distances_df["duration(hhmmss)"] = pd.to_datetime(distances_df["duration(s)"], unit="s").dt.strftime("%H:%M:%S")

    distances_df["fixture_key"] = (
        distances_df["destination_name"].str.strip() + "-" + distances_df["origin_name"].str.strip().astype(str)
    )

    return distances_df

In [8]:
journeys_df = create_distances_df()
journeys_df

,origin,destination,duration(s),distance(m),origin_name,origin_league,destination_name,destination_league,distance(miles),duration(hhmmss),fixture_key
0,92,93,13991.7,324543.4,Aldershot Town,National League,Altrincham,National League,201.661857,03:53:11,Altrincham-Aldershot Town
1,94,93,11958.8,286893.0,Boreham Wood,National League,Altrincham,National League,178.266990,03:19:18,Altrincham-Boreham Wood
2,95,93,10253.8,203348.5,Boston United,National League,Altrincham,National League,126.354861,02:50:53,Altrincham-Boston United
3,96,93,8981.9,210451.2,Brackley Town,National League,Altrincham,National League,130.768273,02:29:41,Altrincham-Brackley Town
4,97,93,14012.8,339844.4,Braintree Town,National League,Altrincham,National League,211.169455,03:53:32,Altrincham-Braintree Town
...,...,...,...,...,...,...,...,...,...,...,...
547,111,92,9286.5,211953.7,Tamworth,National League,Aldershot Town,National League,131.701883,02:34:46,Aldershot Town-Tamworth
548,112,92,12907.8,292858.8,Truro City,National League,Aldershot Town,National League,181.973965,03:35:07,Aldershot Town-Truro City
549,113,92,2824.2,57931.2,Wealdstone,National League,Aldershot Town,National League,35.996768,00:47:04,Aldershot Town-Wealdstone
550,114,92,1484.9,18625.8,Woking,National League,Aldershot Town,National League,11.573532,00:24:44,Aldershot Town-Woking
